# Function Questions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

## Que1: Data Type Conversions and Casting

**Difficulty:** Easy

### Problem

You have imported raw data where all columns are strings. Some values are malformed (N/A, empty strings, "null", invalid formats). You need to convert each column to its proper data type while gracefully handling errors.

**Schema columns:** `raw_imports.id`, `raw_imports.date_str`, `raw_imports.amount_str`, `raw_imports.quantity_str`, `raw_imports.is_active_str`

**Output columns:** `id`, `date_val`, `amount_val`, `quantity_val`, `is_active`

Sort by the first column in ascending order.

### Examples

#### Example 1

**Input:**

**raw_imports:**

| id | date_str | amount_str | quantity_str | is_active_str |
|---:|----------|-----------:|-------------:|---------------|
| 1 | 2024-01-15 | 100.50 | 5 | true |
| 2 | 2024-01-16 | 250.75 | 10 | false |
| 3 | 2024-01-17 | | 8 | true |
| 4 | 2024-01-18 | 150.25 | | false |
| 5 | 2024-01-19 | 200.00 | | true |

**Output:**

| id | date_val | amount_val | quantity_val | is_active |
|---:|----------|----------:|-------------:|----------:|
| 1 | 2024-01-15 | 100.5 | 5.0 | 1.0 |
| 2 | 2024-01-16 | 250.75 | 10.0 | 0.0 |
| 3 | 2024-01-17 | NULL | 8.0 | 1.0 |
| 4 | 2024-01-18 | 150.25 | NULL | 0.0 |
| 5 | 2024-01-19 | 200.0 | NULL | 1.0 |

**Explanation:** The output is derived by applying the required transformations to the input data.

### Constraints

- Dates: YYYY-MM-DD format (string output).
- Amounts: rounded to 2 decimal places.
- Quantities: integers (no decimal point).
- Boolean: 1 (true) or 0 (false).
- NULL if malformed.
- Invalid/missing values: NULL.
- Order by id ASC.

In [0]:
raw_imports_data = [(1,"2024-01-15","100.50","5","true"),(2,"2024-01-16","250.75","10","false"),(3,"2024-01-17",None,"8","true"),(4,"2024-01-18","150.25",None,"false"),(5,"2024-01-19","200.00",None,"true")]
raw_imports_df = spark.createDataFrame(raw_imports_data, ["id","date_str","amount_str","quantity_str","is_active_str"])

output_df = (
raw_imports_df.
    withColumn("date", to_date(col("date_str"), 'yyyy-MM-dd')).
    withColumn("amount_val", col("amount_str").cast("double")).
    withColumn("quantity_val", col("quantity_str").cast("double")).
    withColumn("is_active_val", 
        when(lower(col("is_active_str")) == 'true', 1).otherwise(0)
    )
    .drop("amount_str", "date_str", "is_active_str", "quantity_str")
    .orderBy("id")
)



display(output_df)

## Que2: E-Commerce Date Dimension Analysis

**Difficulty:** Medium

### Problem

Flag weekend orders from a feed with untrusted date strings. You are a data engineer at Amazon. The order feed stores `order_date` as raw text, and upstream systems occasionally write malformed values. Marketing wants to know which orders were placed on weekends, broken out by product category, using only the rows whose dates can be trusted.

Write a query that keeps only the rows of `ecom_dt_orders` whose `order_date` exactly matches the MM/DD/YYYY pattern (two digits, slash, two digits, slash, four digits); discard any row that does not match. Parse the valid dates and join the surviving orders to `ecom_dt_products` on `product_id` (inner join). For each row, compute `is_weekend` = 1 if the parsed date falls on a Saturday or Sunday, otherwise 0. Return `order_date` formatted back as MM/DD/YYYY text. Sort the results by category ascending, then by the parsed order date descending.

**Schema columns:** `ecom_dt_orders.order_id`, `ecom_dt_orders.product_id`, `ecom_dt_orders.user_id`, `ecom_dt_orders.order_date`, `ecom_dt_products.product_id`, `ecom_dt_products.product_name`, `ecom_dt_products.category`

**Output columns:** `category`, `is_weekend`, `order_date`, `product_name`, `user_id`

### Examples

#### Example 1

**Input:**

**ecom_dt_orders:**

| order_id | product_id | user_id | order_date |
|---------:|-----------|---------|------------|
| 1 | P001 | U001 | 02/25/2023 |
| 2 | P002 | U001 | 03/14/2023 |
| 3 | P001 | U002 | 03/16/2023 |
| 4 | P003 | U002 | 03/18/2023 |
| 5 | P004 | U003 | 04/01/2023 |

**ecom_dt_products:**

| product_id | product_name | category |
|-----------|--------------|----------|
| P001 | Product 1 | Electronics |
| P002 | Product 2 | Clothing |
| P003 | Product 3 | Home Goods |
| P004 | Product 4 | Books |

**Output:**

| category | is_weekend | order_date | product_name | user_id |
|----------|:----------:|------------|--------------|--------|
| Books | 1 | 04/01/2023 | Product 4 | U003 |
| Clothing | 0 | 03/14/2023 | Product 2 | U001 |
| Electronics | 0 | 03/16/2023 | Product 1 | U002 |
| Electronics | 1 | 02/25/2023 | Product 1 | U001 |
| Home Goods | 1 | 03/18/2023 | Product 3 | U002 |

**Explanation:** 04/01/2023 is a Saturday, so the Books order gets `is_weekend` = 1, while 03/14/2023 (a Tuesday) gets 0. Within Electronics, 03/16/2023 appears before 02/25/2023 because rows are sorted by category ascending then by parsed date descending.

### Constraints

- Keep only rows where `order_date` exactly matches the MM/DD/YYYY pattern; discard malformed dates.
- Inner join the surviving orders to `ecom_dt_products` on `product_id`.
- `is_weekend` is 1 when the parsed date falls on a Saturday or Sunday, otherwise 0.
- Output `order_date` formatted back as MM/DD/YYYY text.
- Output columns must be exactly `category`, `is_weekend`, `order_date`, `product_name`, `user_id`.
- Sort by category ascending, then by the parsed order date descending.

In [0]:
ecom_dt_orders_data = [(1,"P001","U001","02/25/2023"),(2,"P002","U001","03-14-2023"),(3,"P001","U002","03/16/2023"),(4,"P003","U002","03/18/2023"),(5,"P004","U003","04/01/2023")]
ecom_dt_orders_df = spark.createDataFrame(ecom_dt_orders_data, ["order_id","product_id","user_id","order_date"])
ecom_dt_products_data = [("P001","Product 1","Electronics"),("P002","Product 2","Clothing"),("P003","Product 3","Home Goods"),("P004","Product 4","Books")]
ecom_dt_products_df = spark.createDataFrame(ecom_dt_products_data, ["product_id","product_name","category"])

ecom_dt_orders_df = (
ecom_dt_orders_df
    .withColumn("valid_date", try_to_date(col("order_date"), "MM/dd/yyyy"))
    .withColumn("is_weekend", 
        when(dayofweek(col("valid_date")).isin([1, 7]), 1).otherwise(0)
    )
    .filter(col("valid_date").isNotNull())
)

display(ecom_dt_orders_df)
display(ecom_dt_products_df)

joined_df = (
    ecom_dt_orders_df
    .join(ecom_dt_products_df, on="product_id", how="inner")
    .select("category","order_date","is_weekend","product_name","user_id")
    .orderBy(col("category"), col("order_date").desc())
)

display(joined_df)

## Que3: Ride Request to Completion Time by Hour

**Difficulty:** Medium

### Problem

A ride-hailing team wants fulfillment times summarized by the hour in which each ride was requested. For each `hour_of_day`, return `ride_count`, `avg_total_minutes` from request to drop-off, `avg_wait_minutes` from request to pickup, and `avg_ride_minutes` from pickup to drop-off; round all three averages to 1 decimal place. Order the result by hour ascending.

**Schema columns:** `rides.ride_id`, `rides.request_time`, `rides.pickup_time`, `rides.dropoff_time`

**Output columns:** `hour_of_day`, `ride_count`, `avg_total_minutes`, `avg_wait_minutes`, `avg_ride_minutes`

### Examples

#### Example 1

**Input:**

**rides:**

| ride_id | request_time | pickup_time | dropoff_time |
|--------:|----------------------|---------------------|---------------------|
| 1 | 2026-01-01T09:00:00 | 2026-01-01T09:10:00 | 2026-01-01T09:30:00 |
| 2 | 2026-01-01T09:20:00 | 2026-01-01T09:40:00 | 2026-01-01T10:10:00 |
| 3 | 2026-01-01T10:05:00 | 2026-01-01T10:10:00 | 2026-01-01T10:25:00 |
| 4 | 2026-01-02T09:05:00 | 2026-01-02T09:20:00 | 2026-01-02T09:45:00 |

**Output:**

| hour_of_day | ride_count | avg_total_minutes | avg_wait_minutes | avg_ride_minutes |
|------------:|-----------:|------------------:|-----------------:|-----------------:|
| 9 | 3 | 40.0 | 15.0 | 25.0 |
| 10 | 1 | 20.0 | 5.0 | 15.0 |

**Explanation:** The three 09:00-hour rides take 30, 50, and 40 minutes from request to drop-off, producing `ride_count` 3 and `avg_total_minutes` 40.0; their waits average (10 + 20 + 15) / 3 = 15.0 minutes.

### Constraints

- Use every supplied input row when calculating the result.
- Preserve the calculation, filtering, tie handling, and ordering described in the problem.
- Return results matching the expected output schema and order.

In [0]:
rides_data = [(1,"2026-01-01T09:00:00","2026-01-01T09:10:00","2026-01-01T09:30:00"),(2,"2026-01-01T09:20:00","2026-01-01T09:40:00","2026-01-01T10:10:00"),(3,"2026-01-01T10:05:00","2026-01-01T10:10:00","2026-01-01T10:25:00"),(4,"2026-01-02T09:05:00","2026-01-02T09:20:00","2026-01-02T09:45:00")]
rides_df = spark.createDataFrame(rides_data, ["ride_id","request_time","pickup_time","dropoff_time"])

display(rides_df)

rides_df = (
    rides_df
        .withColumn("request_time", to_timestamp(col("request_time"), "yyyy-MM-dd'T'HH:mm:ss"))
        .withColumn("pickup_time", to_timestamp(col("pickup_time"), "yyyy-MM-dd'T'HH:mm:ss"))
        .withColumn("dropoff_time", to_timestamp(col("dropoff_time"), "yyyy-MM-dd'T'HH:mm:ss"))
        .withColumn("hour_of_day", hour(col("request_time")))
        .withColumn("total_time", timestamp_diff("minute", col("request_time"), col("dropoff_time")))
        .withColumn("wait_time", timestamp_diff("minute", col("request_time"), col("pickup_time")))
        .withColumn("ride_time", timestamp_diff("minute", col("pickup_time"), col("dropoff_time")))       
)

display(rides_df)

output_df = (
rides_df.groupBy("hour_of_day").agg(
    round(avg(col("total_time")), 2).alias("avg_total_time"),
    round(avg(col("wait_time")), 2).alias("avg_wait_time"),
    round(avg(col("ride_time")),  2).alias("avg_ride_time")
)
)

display(output_df)


## Que4: Customers with Purchases on Multiple Distinct Days

**Difficulty:** Medium

### Problem

Amazon wants to find its loyal customers: those who placed orders on several separate occasions. For every customer who shopped on at least 3 different calendar dates, report the customer, how many distinct dates they ordered on, the earliest such date, and the latest such date. Two or more orders placed on the same date count as a single purchase day.

**Schema columns:** `orders.order_id`, `orders.customer_id`, `orders.order_date`, `orders.amount`

**Output columns:** `customer_id`, `distinct_days`, `first_purchase`, `last_purchase`

Sort the results by `customer_id`.

### Examples

#### Example 1

**Input:**

**orders:**

| order_id | customer_id | order_date | amount |
|---------:|------------:|------------|-------:|
| 1001 | 101 | 2024-01-05 | 150 |
| 1002 | 101 | 2024-01-05 | 200 |
| 1003 | 101 | 2024-01-10 | 75.5 |
| 1004 | 101 | 2024-02-15 | 120 |
| 1005 | 102 | 2024-01-08 | 300 |
| 1007 | 102 | 2024-02-20 | 180 |
| 1009 | 104 | 2024-01-15 | 450 |
| 1010 | 104 | 2024-01-20 | 320 |
| 1012 | 104 | 2024-03-10 | 275 |

**Output:**

| customer_id | distinct_days | first_purchase | last_purchase |
|------------:|--------------:|----------------|---------------|
| 101 | 3 | 2024-01-05 | 2024-02-15 |
| 104 | 3 | 2024-01-15 | 2024-03-10 |

**Explanation:** Customer 101 has four orders but two share the date 2024-01-05, so they land on 3 distinct dates, which meets the threshold. Customer 102 ordered on only 2 distinct dates and is excluded.

### Constraints

- Include only customers who ordered on at least 3 distinct dates.
- Multiple orders on the same date count as one purchase day.
- `first_purchase` and `last_purchase` are the earliest and latest order dates, formatted as YYYY-MM-DD.
- Sort by `customer_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
orders_data = [(1001,101,"2024-01-05",150),(1002,101,"2024-01-05",200),(1003,101,"2024-01-10",75),(1004,101,"2024-02-15",120),(1005,102,"2024-01-08",300),(1007,102,"2024-02-20",180),(1009,104,"2024-01-15",450),(1010,104,"2024-01-20",320),(1012,104,"2024-03-10",275)]
orders_df = spark.createDataFrame(orders_data, ["order_id","customer_id","order_date","amount"])

display(orders_df)

grouped_df = (
orders_df.groupBy("customer_id").agg(
    countDistinct("order_date").alias("distinct_days"),
    min("order_date").alias("first_purchase"),
    max("order_date").alias("last_purchase")
)
.filter(col("distinct_days") >= 3)
.orderBy("customer_id")
)

display(grouped_df)

## Que5: Column Selection and Reordering with Derived Metrics

**Difficulty:** Medium

### Problem

You have a wide web analytics table with many columns. You need to extract key business metrics, compute derived columns like session duration and conversion rate, and present them in a specific order.

**Schema columns:** `web_analytics.session_id`, `web_analytics.user_id`, `web_analytics.page_url`, `web_analytics.referrer`, `web_analytics.device_type`, `web_analytics.browser`, `web_analytics.os`, `web_analytics.country`, `web_analytics.city`, `web_analytics.session_start`, `web_analytics.session_end`, `web_analytics.page_views`, `web_analytics.clicks`, `web_analytics.conversions`, `web_analytics.bounce`

**Output columns:** `session_id`, `user_id`, `device_type`, `country`, `page_views`, `clicks`, `conversions`, `session_duration_minutes`, `conversion_rate`

Sort by the first column in ascending order.

### Examples

#### Example 1

**Input:**

**web_analytics:**

| session_id | user_id | device_type | country | page_views | clicks | conversions | session_start | session_end |
|-----------|--------|------------|---------|----------:|-------:|------------:|----------------------|---------------------|
| S001 | U101 | mobile | US | 5 | 12 | 1 | 2024-01-15 10:00:00 | 2024-01-15 10:15:00 |
| S002 | U102 | desktop | US | 2 | 3 | 0 | 2024-01-15 11:00:00 | 2024-01-15 11:08:00 |
| S003 | U103 | mobile | UK | 8 | 25 | 2 | 2024-01-15 12:00:00 | 2024-01-15 12:20:00 |
| S004 | U104 | desktop | US | 3 | 8 | 1 | 2024-01-15 13:00:00 | 2024-01-15 13:10:00 |
| S005 | U105 | mobile | CA | 1 | 2 | 0 | 2024-01-15 14:00:00 | 2024-01-15 14:05:00 |

**Output:**

| session_id | user_id | device_type | country | page_views | clicks | conversions | session_duration_minutes | conversion_rate |
|-----------|--------|------------|---------|----------:|-------:|------------:|------------------------:|----------------:|
| S001 | U101 | mobile | US | 5 | 12 | 1 | 15.0 | 0.2 |
| S002 | U102 | desktop | US | 2 | 3 | 0 | 8.0 | 0.0 |
| S003 | U103 | mobile | UK | 8 | 25 | 2 | 20.0 | 0.25 |
| S004 | U104 | desktop | US | 3 | 8 | 1 | 10.0 | 0.33 |
| S005 | U105 | mobile | CA | 1 | 2 | 0 | 5.0 | 0.0 |

**Explanation:** The output is derived by applying the required transformations to the input data.

### Constraints

- `session_duration_minutes` must be calculated in minutes (not seconds).
- `conversion_rate` = conversions / page_views (round to 2 decimal places).
- If `page_views` = 0, `conversion_rate` = 0.0.
- Results ordered by `session_id` alphabetically.

In [0]:
web_analytics_data = [("S001","U101","/products","google","mobile","Chrome","iOS","US","New York","2024-01-15 10:00:00","2024-01-15 10:15:00",5,12,1,0),("S002","U102","/home","direct","desktop","Firefox","Windows","US","San Francisco","2024-01-15 11:00:00","2024-01-15 11:08:00",2,3,0,1),("S003","U103","/products","facebook","mobile","Safari","iOS","UK","London","2024-01-15 12:00:00","2024-01-15 12:20:00",8,25,2,0),("S004","U104","/checkout","google","desktop","Chrome","Windows","US","Los Angeles","2024-01-15 13:00:00","2024-01-15 13:10:00",3,8,0,0),("S005","U105","/home","direct","mobile","Chrome","Android","CA","Toronto","2024-01-15 14:00:00","2024-01-15 14:05:00",1,2,0,1)]
web_analytics_df = spark.createDataFrame(web_analytics_data, ["session_id","user_id","page_url","referrer","device_type","browser","os","country","city","session_start","session_end","page_views","clicks","conversions","bounce"])

web_analytics_df = (
web_analytics_df
    .withColumn("session_start", to_timestamp(col("session_start"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("session_end", to_timestamp(col("session_end"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("duration", timestamp_diff("minute", col("session_start"), col("session_end")))
    .withColumn("conversion_rate", round(col("conversions") / col("page_views"), 2))
)

display(web_analytics_df)

## Que6: Regex Pattern Matching

**Difficulty:** Medium

### Problem

You are a data engineer at LinkedIn working on an ingestion pipeline that receives free-form text strings. Some strings are clean arithmetic expressions that downstream jobs can evaluate; the rest are noise, and you must filter the feed down to only the valid expressions before anything is evaluated.

Write a query that returns the `id` and `expression` of every row in `df_math_expr` whose expression is a valid arithmetic expression: it consists only of non-negative integers and the four operators +, -, *, /, it must start and end with an integer, and consecutive integers must be separated by exactly one operator. Strings containing letters, spaces, decimal points, or any other character are invalid. Return qualifying rows in ascending id order.

**Schema columns:** `df_math_expr.id`, `df_math_expr.expression`

**Output columns:** `id`, `expression`

### Examples

#### Example 1

**Input:**

**df_math_expr:**

| id | expression |
|---:|-----------|
| 1 | 5+3 |
| 2 | hello |
| 3 | 6/3 |
| 4 | world |
| 5 | 2*3+1 |

**Output:**

| id | expression |
|---:|-----------|
| 1 | 5+3 |
| 3 | 6/3 |
| 5 | 2*3+1 |

**Explanation:** Rows 1, 3, and 5 are made up solely of integers separated by single arithmetic operators, so they qualify. Rows 2 and 4 contain letters, so they are filtered out.

### Constraints

- Valid expressions contain only non-negative integers and the characters +, -, *, /.
- An expression must start and end with an integer.
- Exactly one operator must separate consecutive integers.
- Letters, spaces, decimal points, or any other characters make an expression invalid.
- Return both columns `id` and `expression` unchanged.
- Sort by `id` in ascending order.

In [0]:
df_math_expr_data = [(1,"5+3"),(2,"hello"),(3,"6/3"),(4,"world"),(5,"2*3+1")]
df_math_expr_df = spark.createDataFrame(df_math_expr_data, ["id","expression"])

display(df_math_expr_df)

result = (
    df_math_expr_df
    .filter(
        col("expression").rlike(r"^[0-9]+([+\-*/][0-9]+)*$")
    )
    .select("id", "expression")
    .orderBy("id")
)

display(result)

## Que7: Shortest Distance Between Two Points

**Difficulty:** Medium

### Problem

Two points on a 2D plane are defined as follows:
- Point 1: p1(x1, y1) where x1 is the minimum Northern Latitude (`LAT_N`) and y1 is the minimum Western Longitude (`LONG_W`) from the STATION table.
- Point 2: p2(x2, y2) where x2 is the maximum Northern Latitude (`LAT_N`) and y2 is the maximum Western Longitude (`LONG_W`) from the STATION table.

Write a query to calculate the Euclidean distance between these two points and display the result rounded to 4 decimal places.

**Schema columns:** `dpd_plane_dis.ID`, `dpd_plane_dis.CITY`, `dpd_plane_dis.STATE`, `dpd_plane_dis.LAT_N`, `dpd_plane_dis.LONG_W`

**Output columns:** `Euclidean Distance`

### Examples

#### Example 1

**Input:**

**dpd_plane_dis:**

| ID | CITY | STATE | LAT_N | LONG_W |
|---:|------|-------|------:|-------:|
| 1 | CityA | State1 | 23.5 | 78.5 |
| 2 | CityB | State2 | 45.3 | 89.6 |
| 3 | CityC | State3 | 12.4 | 67.4 |
| 4 | CityD | State4 | 56.7 | 92.3 |

**Output:**

| Euclidean Distance |
|-------------------:|
| 50.8183 |

### Constraints

- Handle NULL values appropriately.
- Return results matching the expected output schema and order.

In [0]:
dpd_plane_dis_data = [(1,"CityA","State1",23.5,78.5),(2,"CityB","State2",45.3,89.6),(3,"CityC","State3",12.4,67.4),(4,"CityD","State4",56.7,92.3)]
dpd_plane_dis_df = spark.createDataFrame(dpd_plane_dis_data, ["ID","CITY","STATE","LAT_N","LONG_W"])

display(dpd_plane_dis_df)

result = (
    dpd_plane_dis_df
    .agg(
        min("LAT_N").alias("min_lat"),
        max("LAT_N").alias("max_lat"),
        min("LONG_W").alias("min_long"),
        max("LONG_W").alias("max_long")
    )
    .select(
        round(
            sqrt(
                pow(col("max_lat") - col("min_lat"), 2) +
                pow(col("max_long") - col("min_long"), 2)
            ),
            4
        ).alias("Euclidean Distance")
    )
)

display(result)

## Que8: Monthly Distinct User Comparison

**Difficulty:** Medium

### Problem

Identify all email records from days where the number of distinct users receiving emails is greater than the number of distinct users sending emails.

**Schema columns:** `duc_distinct_user.id`, `duc_distinct_user.from_user`, `duc_distinct_user.to_user`, `duc_distinct_user.day`

**Output columns:** `id`, `from_user`, `to_user`, `day`

### Examples

#### Example 1

**Input:**

**duc_distinct_user:**

| id | from_user | to_user | day |
|---:|-----------|---------|----:|
| 0 | 6edf0be4b2267df1fa | 75d295377a46f832 | 3610 |
| 1 | 6edf0be4b2267df1fa | 32ded68d89443e80 | 86 |
| 2 | 6edf0be4b2267df1fa | 55e60cfcc9dc49c1 | 7e10 |
| 3 | 6edf0be4b2267df1fa | e0e0defbb9ec47f6 | f76 |

**Output:**

| id | from_user | to_user | day |
|---:|-----------|---------|----:|
| 0 | 6edf0be4b2267df1fa | 75d295377a46f832 | 3610 |
| 1 | 6edf0be4b2267df1fa | 32ded68d89443e80 | 86 |
| 2 | 6edf0be4b2267df1fa | 55e60cfcc9dc49c1 | 7e10 |
| 3 | 6edf0be4b2267df1fa | e0e0defbb9ec47f6 | f76 |

### Constraints

- Handle NULL values appropriately.
- Return results matching the expected output schema and order.

In [0]:
duc_distinct_user_data = [(0,"6edf0be4b2267df1fa","75d295377a46f832","3610"),(1,"6edf0be4b2267df1fa","32ded68d89443e80","86"),(2,"6edf0be4b2267df1fa","55e60cfcc9dc49c1","7e10"),(3,"6edf0be4b2267df1fa","e0e0defbb9ec47f6","f76")]
duc_distinct_user_df = spark.createDataFrame(duc_distinct_user_data, ["id","from_user","to_user","day"])

display(duc_distinct_user_df)

day_counts = (
    duc_distinct_user_df
    .groupBy("day")
    .agg(
        countDistinct("from_user").alias("distinct_senders"),
        countDistinct("to_user").alias("distinct_receivers")
    )
)

result = (
    duc_distinct_user_df
    .join(
        day_counts.filter(
            col("distinct_receivers") > col("distinct_senders")
        ).select("day"),
        on="day",
        how="inner"
    )
    .select("id", "from_user", "to_user", "day")
)

display(result)

## Que9: Sales Performance Classification

**Difficulty:** Medium

### Problem

An advertising team wants to classify each product by its total units sold. Label totals of at least 30 `Outstanding`, 20–29 `Satisfactory`, 10–19 `Unsatisfactory`, 1–9 `Poor`, and 0 `No Sales`. Return `product_id`, `total_units_sold`, and `ad_performance`, ordered by total units descending.

**Schema columns:** `sct_sales_classification.user_id`, `sct_sales_classification.created_at`, `sct_sales_classification.product_id`, `sct_sales_classification.quantity`, `sct_sales_classification.price`

**Output columns:** `product_id`, `total_units_sold`, `ad_performance`

### Examples

#### Example 1

**Input:**

**sct_sales_classification:**

| price | user_id | quantity | created_at | product_id |
|------:|--------:|---------:|------------|----------:|
| 200 | 1 | 25 | 2020-01-01 | 101 |
| 150 | 2 | 5 | 2020-01-01 | 102 |
| 300 | 3 | 15 | 2020-01-02 | 103 |
| 200 | 4 | 10 | 2020-01-03 | 101 |
| 150 | 5 | 22 | 2020-01-04 | 102 |
| 120 | 6 | 8 | 2020-01-05 | 104 |
| 250 | 7 | 18 | 2020-01-06 | 105 |
| 200 | 8 | 30 | 2020-01-07 | 101 |

**Output:**

| product_id | total_units_sold | ad_performance |
|-----------:|-----------------:|----------------|
| 101 | 65 | Outstanding |
| 102 | 27 | Satisfactory |
| 105 | 18 | Unsatisfactory |
| 103 | 15 | Unsatisfactory |
| 104 | 8 | Poor |

**Explanation:** Product 101 sells 25 + 10 + 30 = 65 units in total, which meets the 30-or-more threshold and is classified as Outstanding.

### Constraints

- Use every supplied input row when calculating the result.
- Preserve the calculation, filtering, tie handling, and ordering described in the problem.
- Return results matching the expected output schema and order.

In [0]:
sct_sales_classification_data = [(200,1,25,"2020-01-01",101),(150,2,5,"2020-01-01",102),(300,3,15,"2020-01-02",103),(200,4,10,"2020-01-03",101),(150,5,22,"2020-01-04",102),(120,6,8,"2020-01-05",104),(250,7,18,"2020-01-06",105),(200,8,30,"2020-01-07",101)]
sct_sales_classification_df = spark.createDataFrame(sct_sales_classification_data, ["price","user_id","quantity","created_at","product_id"])

display(sct_sales_classification_df)

grouped_df = (
sct_sales_classification_df.groupBy("product_id").agg(
    sum("quantity").alias("total_units_sold")
))

# Outstanding, 20–29 Satisfactory, 10–19 Unsatisfactory, 1–9 Poor, and 0 No Sales. Return product_id, total_units_sold, and ad_performance, ordered by total units descending.

output_df = (
grouped_df
    .withColumn("ad_performance",
        when(col("total_units_sold") >= 20, lit("Outstanding"))
        .when(col("total_units_sold") >= 19, lit("Satisfactory"))
        .when(col("total_units_sold") >= 1, lit("Unsatisfactory"))
        .otherwise(lit("Poor"))
    )
    .orderBy(col("total_units_sold").desc())
)

display(output_df)




## Que10: Gender-Based Salary Comparison

**Difficulty:** Medium

### Problem

In a marathon, two types of times are recorded — Gun time (starts when the race officially begins) and Net time (starts when a runner crosses the starting line). Determine if the time difference between the gun time and net time varies between male and female runners.

**Schema columns:** `gc_marathon_female.age`, `gc_marathon_female.div_tot`, `gc_marathon_female.gun_time`, `gc_marathon_female.hometown`, `gc_marathon_female.net_time`, `gc_marathon_female.num`, `gc_marathon_female.pace`, `gc_marathon_female.person_name`, `gc_marathon_female.place`, `gc_marathon_male.age`, `gc_marathon_male.div_tot`, `gc_marathon_male.gun_time`, `gc_marathon_male.hometown`, `gc_marathon_male.net_time`, `gc_marathon_male.num`, `gc_marathon_male.pace`, `gc_marathon_male.person_name`, `gc_marathon_male.place`

**Output columns:** `Gender`, `Avg_Abs_Diff`

Sort the results by `CASE WHEN gender = 'Male' THEN 1 ELSE 2 END`.

### Examples

#### Example 1

**Input:**

**gc_marathon_female:**

| age | div_tot | gun_time | hometown | net_time | num | pace | person_name | place |
|----:|---------|--------:|----------|--------:|----:|-----:|-------------|------:|
| 28 | 1/100 | 3650 | San Francisco | 3600 | 20 | 1510 | Jane Doe | 1 |
| 26 | 2/100 | 3900 | Los Angeles | 3850 | 20 | 2530 | Emily Davis | 2 |
| 24 | 3/100 | 4100 | Seattle | 4050 | 20 | 3590 | Anna Brown | 3 |

**gc_marathon_male:**

| age | div_tot | gun_time | hometown | net_time | num | pace | person_name | place |
|----:|---------|--------:|----------|--------:|----:|-----:|-------------|------:|
| 25 | 1/100 | 3600 | New York | 3400 | 10 | 1500 | John Doe | 1 |
| 30 | 2/100 | 4000 | Boston | 3850 | 10 | 2550 | Michael Smith | 2 |
| 22 | 3/100 | 4200 | Chicago | 4150 | 10 | 3600 | David Johnson | 3 |

**Output:**

| Gender | Avg_Abs_Diff |
|--------|-------------:|
| Male | 133.33 |
| Female | 50 |

### Constraints

- Handle NULL values appropriately.
- Return results matching the expected output schema and order.

In [0]:
gc_marathon_female_data = [(28,"1/100",3650,"San Francisco",3600,20,1510,"Jane Doe",1),(26,"2/100",3900,"Los Angeles",3850,20,2530,"Emily Davis",2),(24,"3/100",4100,"Seattle",4050,20,3590,"Anna Brown",3)]
gc_marathon_female_df = spark.createDataFrame(gc_marathon_female_data, ["age","div_tot","gun_time","hometown","net_time","num","pace","person_name","place"])

gc_marathon_male_data = [(25,"1/100",3600,"New York",3400,10,1500,"John Doe",1),(30,"2/100",4000,"Boston",3850,10,2550,"Michael Smith",2),(22,"3/100",4200,"Chicago",4150,10,3600,"David Johnson",3)]
gc_marathon_male_df = spark.createDataFrame(gc_marathon_male_data, ["age","div_tot","gun_time","hometown","net_time","num","pace","person_name","place"])

display(gc_marathon_female_df)
display(gc_marathon_male_df)


grouped_df_male = (
gc_marathon_male_df
    .select(round(avg(col("gun_time") - col("net_time")),2).alias("Avg_Abf_Diff"))
    .withColumn("gender", lit("Male"))
    .select("gender", "Avg_Abf_Diff")
)

grouped_df_female = (
gc_marathon_female_df
    .select(round(avg(col("gun_time") - col("net_time")),2).alias("Avg_Abf_Diff"))
    .withColumn("gender", lit("Female"))
    .select("gender", "Avg_Abf_Diff")
)

output_df = grouped_df_male.union(grouped_df_female)

display(output_df)


## Que11: Monthly Sales Trend Analysis

**Difficulty:** Medium

### Problem

Write a query to summarize transaction data by month and country. For each month and country, calculate the total number of transactions (`trans_count`) and the total transaction amount (`trans_total_amount`).

**Schema columns:** `ma_transaction.id`, `ma_transaction.country`, `ma_transaction.state`, `ma_transaction.amount`, `ma_transaction.trans_date`

**Output columns:** `month`, `country`, `trans_count`, `approved_count`, `trans_total_amount`, `approved_total_amount`

Sort the results by `month`, `country`.

### Examples

#### Example 1

**Input:**

**ma_transaction:**

| id | country | state | amount | trans_date |
|---:|---------|----------|-------:|------------|
| 121 | US | approved | 1000 | 2018-12-18 |
| 122 | US | declined | 2000 | 2018-12-19 |
| 123 | US | approved | 2000 | 2019-01-01 |
| 124 | DE | approved | 2000 | 2019-01-07 |

**Output:**

| month | country | trans_count | approved_count | trans_total_amount | approved_total_amount |
|-------|---------|------------:|---------------:|-------------------:|----------------------:|
| 2018-12 | US | 2 | 1 | 3000 | 1000 |
| 2019-01 | DE | 1 | 1 | 2000 | 2000 |
| 2019-01 | US | 1 | 1 | 2000 | 2000 |

### Constraints

- Handle NULL values appropriately.
- Return results matching the expected output schema and order.

In [0]:
ma_transaction_data = [(121,"US","approved",1000,"2018-12-18"),(122,"US","declined",2000,"2018-12-19"),(123,"US","approved",2000,"2019-01-01"),(124,"DE","approved",2000,"2019-01-07")]
ma_transaction_df = spark.createDataFrame(ma_transaction_data, ["id","country","state","amount","trans_date"])

ma_transaction_df = ma_transaction_df.withColumn("trans_date", to_date(col("trans_date"), "yyyy-MM-dd")).withColumn("month", date_format("trans_date", "yyyy-MM"))

display(ma_transaction_df)

output_df = (
ma_transaction_df.groupBy("month", "country").agg(
    count("id").alias("trans_count"),
    count(when(col("state") == "approved", "id")).alias("approved_count"),
    sum("amount").alias("trans_total_amount"),
    sum(when(col("state") == "approved", col("amount")).otherwise(0)).alias("approved_total_amount")
)
.orderBy("month", "country")
)
display(output_df)


## Que12: Data Pipeline Audit Table Query

**Difficulty:** Easy

### Problem

Every execution of your ETL pipelines writes an audit row to `pipeline_runs` with the start and end timestamps, a status, and the number of rows processed. The platform team wants a daily reliability report per pipeline to spot failure spikes and runtime regressions.

For each pipeline and each calendar day (the date part of `start_time`), return the `pipeline_name`, the day as `run_date`, `total_runs` (every run that started that day), `success_count` (how many of those runs have a status of exactly `success`), and `avg_duration_minutes` (the average run duration in minutes, where a run's duration is `end_time` minus `start_time`, rounded to 2 decimal places). A run whose `end_time` is missing still counts toward `total_runs` but is left out of the duration average. Sort by `pipeline_name` ascending, then by `run_date` ascending.

**Schema columns:** `pipeline_runs.run_id`, `pipeline_runs.pipeline_name`, `pipeline_runs.start_time`, `pipeline_runs.end_time`, `pipeline_runs.status`, `pipeline_runs.rows_processed`

**Output columns:** `pipeline_name`, `run_date`, `total_runs`, `success_count`, `avg_duration_minutes`

### Examples

#### Example 1

**Input:**

**pipeline_runs:**

| run_id | pipeline_name | start_time | end_time | status | rows_processed |
|-------:|---------------|---------------------|---------------------|---------|---------------:|
| 1 | etl_sales | 2024-01-01 00:00:00 | 2024-01-01 00:05:00 | success | 5000 |
| 2 | etl_sales | 2024-01-01 06:00:00 | 2024-01-01 06:05:20 | success | 5100 |
| 5 | etl_customers | 2024-01-02 00:00:00 | 2024-01-02 00:02:40 | failed | 2100 |
| 6 | etl_customers | 2024-01-02 06:00:00 | 2024-01-02 06:02:35 | success | 2050 |
| 21 | etl_customers | 2024-01-02 12:00:00 | NULL | running | 0 |

**Output:**

| pipeline_name | run_date | total_runs | success_count | avg_duration_minutes |
|---------------|----------|-----------:|:-------------:|---------------------:|
| etl_customers | 2024-01-02 | 3 | 1 | 2.63 |
| etl_sales | 2024-01-01 | 2 | 2 | 5.17 |

**Explanation:** On 2024-01-02, `etl_customers` started 3 runs, but only run 6 has status `success`. Run 21 has no `end_time`, so only runs 5 and 6 feed the average: their durations of 2 min 40 s and 2 min 35 s average to 2.625 minutes, which rounds to 2.63.

### Constraints

- Each output row summarizes one pipeline on one calendar day, keyed by the date part of `start_time`.
- `success_count` counts only runs whose status is exactly `success`.
- `avg_duration_minutes` is the average of `end_time` minus `start_time` in minutes, rounded to 2 decimal places.
- A run with a missing `end_time` counts in `total_runs` but is excluded from the duration average.
- Return results matching the expected output schema and order.

In [0]:
pipeline_runs_data = [(1,"etl_sales","2024-01-01 00:00:00","2024-01-01 00:05:00","success",5000),(2,"etl_sales","2024-01-01 06:00:00","2024-01-01 06:05:20","success",5100),(5,"etl_customers","2024-01-02 00:00:00","2024-01-02 00:02:40","failed",2100),(6,"etl_customers","2024-01-02 06:00:00","2024-01-02 06:02:35","success",2050),(21,"etl_customers","2024-01-02 12:00:00",None,"running",0)]
pipeline_runs_df = spark.createDataFrame(pipeline_runs_data, ["run_id","pipeline_name","start_time","end_time","status","rows_processed"])


pipeline_runs_df = (
pipeline_runs_df
    .withColumn("start_time", to_timestamp(col("start_time"),  "yyyy-MM-dd HH:mm:ss"))
    .withColumn("end_time", to_timestamp(col("end_time"),  "yyyy-MM-dd HH:mm:ss"))
    .withColumn("run_date", to_date(col("start_time"), "yyyy-MM-dd"))
    .withColumn("duration",  (unix_timestamp(col("end_time")) - unix_timestamp(col("start_time"))) / 60)
)

grouped_df = (
pipeline_runs_df.groupBy("pipeline_name","run_date").agg(
    count(col("run_id")).alias("total_count"),
    count(when(col("status") == "success", 1)).alias("success_count"),
    round(avg(when(col("end_time").isNotNull(),  col("duration"))), 2).alias("avg_duration"),
))


display(grouped_df)



## Que13: Support Ticket Processing Rate

**Difficulty:** Medium

### Problem

A support team groups complaints by type and tracks whether each one was processed. For every type, calculate `processed_rate` as the number of rows whose `processed` value is `True` divided by the total rows of that type, rounded to 2 decimal places.

**Schema columns:** `tpr_ticket_processing.complaint_id`, `tpr_ticket_processing.type`, `tpr_ticket_processing.processed`

**Output columns:** `type`, `processed_rate`

### Examples

#### Example 1

**Input:**

**tpr_ticket_processing:**

| complaint_id | type | processed |
|-------------:|-----:|----------:|
| 0 | 0 | True |
| 1 | 0 | True |
| 2 | 0 | False |
| 3 | 1 | True |
| 4 | 1 | True |
| 5 | 1 | False |
| 6 | 2 | True |
| 7 | 2 | False |
| 8 | 2 | False |

**Output:**

| type | processed_rate |
|-----:|---------------:|
| 0 | 0.67 |
| 1 | 0.67 |
| 2 | 0.33 |

**Explanation:** Types 0 and 1 each process two of three tickets and are ordered by type to break the rate tie.

### Constraints

- Treat the text value `True` as processed and `False` as unprocessed.
- Round `processed_rate` to 2 decimal places.
- Order by `processed_rate` descending, then `type` ascending.
- Return results matching the expected output schema and order.

In [0]:
tpr_ticket_processing_data = [(0,0,"True"),(1,0,"True"),(2,0,"False"),(3,1,"True"),(4,1,"True"),(5,1,"False"),(6,2,"True"),(7,2,"False"),(8,2,"False")]
tpr_ticket_processing_df = spark.createDataFrame(tpr_ticket_processing_data, ["complaint_id","type","processed"])

display(tpr_ticket_processing_df)

output_df = (
tpr_ticket_processing_df.groupBy("type").agg(
    round( count("processed") / count(when(col("processed") == "True", 1))   , 2).alias("processed_rate" )
))

display(output_df)


## Que14: User Defined Functions (UDF) in PySpark

**Difficulty:** Medium

### Problem

E-commerce platforms need to standardize product data for analytics and reporting. Product names contain inconsistent formatting (extra spaces, mixed case), and prices need to be categorized into business tiers for segmentation.

Create a `clean_name` column by normalizing `product_name` (convert to lowercase, strip whitespace), and a `price_tier` column that categorizes products as:
- `budget` for prices < $50
- `mid` for prices between $50–$200 (inclusive of 50, exclusive of 200)
- `premium` for prices > $200

**Schema columns:** `products.product_id`, `products.product_name`, `products.description`, `products.price`

**Output columns:** `product_id`, `clean_name`, `price`, `price_tier`

Order the result by `product_id`.

### Examples

#### Example 1

**Input:**

**products:**

| product_id | product_name | description | price |
|-----------:|--------------|-------------|------:|
| 1 | Apple iPhone 15 | Latest smartphone with A17 Pro chip | 999.99 |
| 2 | Wireless Earbuds | Noise cancelling true wireless earbuds | 79.50 |
| 3 | USB-C Cable | High speed data transfer cable | 25.00 |
| 4 | Samsung 4K TV | 65 inch 4K QLED television | 799.99 |
| 5 | Laptop Stand | Adjustable aluminum laptop stand | 45.99 |

**Output:**

| product_id | clean_name | price | price_tier |
|-----------:|------------|------:|-----------|
| 1 | apple iphone 15 | 999.99 | premium |
| 2 | wireless earbuds | 79.5 | mid |
| 3 | usb-c cable | 25.0 | budget |
| 4 | samsung 4k tv | 799.99 | premium |
| 5 | laptop stand | 45.99 | budget |

**Explanation:** The output is derived by applying the required transformations to the input data according to the problem statement.

### Constraints

- Output must be ordered by `product_id` ascending.
- Prices must be rounded to 2 decimal places.
- `clean_name` must be lowercase with no leading/trailing whitespace.
- Handle edge cases like boundary prices (exactly 50.00, exactly 200.00).

In [0]:
products_data = [(1,"Apple iPhone 15","Latest smartphone with A17 Pro chip",999.99),(2,"Wireless Earbuds","Noise cancelling true wireless earbuds",79.50),(3,"USB-C Cable","High speed data transfer cable",25.00),(4,"Samsung 4K TV","65 inch 4K QLED television",799.99),(5,"Laptop Stand","Adjustable aluminum laptop stand",45.99)]
products_df = spark.createDataFrame(products_data, ["product_id","product_name","description","price"])

display(products_df)

result = (
products_df
    .withColumn("clean_name",lower(trim(col("product_name"))))
    .withColumn("price", round(col("price"), 2))
    .withColumn("price_tier",when(col("price") < 50, "budget")
        .when((col("price") >= 50) & (col("price") < 200), "mid")
        .when(col("price") > 200, "premium")
    )
    .select("product_id","clean_name","price","price_tier")
    .orderBy("product_id")
)

display(result)


## Que15: Rolling 7-Day Active User Count

**Difficulty:** Hard

### Problem

A product team defines a user as active when they perform any recorded action. For each date present in `user_actions`, return the number of distinct users active from 6 days before that date through the date itself as `rolling_7day_active_users`. Order by `action_date` ascending.

**Schema columns:** `user_actions.action_id`, `user_actions.user_id`, `user_actions.action_date`, `user_actions.action_type`

**Output columns:** `action_date`, `rolling_7day_active_users`

### Examples

#### Example 1

**Input:**

**user_actions:**

| user_id | action_id | action_date | action_type |
|--------:|----------:|-------------|-------------|
| 1 | 1 | 2024-01-01 | click |
| 2 | 2 | 2024-01-02 | view |
| 3 | 3 | 2024-01-03 | like |
| 4 | 4 | 2024-01-04 | share |
| 5 | 5 | 2024-01-05 | click |
| 6 | 6 | 2024-01-06 | view |
| 7 | 7 | 2024-01-07 | like |
| 8 | 8 | 2024-01-08 | share |

**Output:**

| action_date | rolling_7day_active_users |
|-------------|-------------------------:|
| 2024-01-01 | 1 |
| 2024-01-02 | 2 |
| 2024-01-03 | 3 |
| 2024-01-04 | 4 |
| 2024-01-05 | 5 |
| 2024-01-06 | 6 |
| 2024-01-07 | 7 |
| 2024-01-08 | 7 |

**Explanation:** For January 8, the inclusive seven-day window is January 2–8; it contains users 2 through 8, so `rolling_7day_active_users` is 7 and user 1 from January 1 is excluded.

### Constraints

- Use every supplied input row when calculating the result.
- Preserve the calculation, filtering, tie handling, and ordering described in the problem.
- Return results matching the expected output schema and order.

In [0]:
user_actions_data = [(1,1,"2024-01-01","click"),(2,2,"2024-01-02","view"),(3,3,"2024-01-03","like"),(4,4,"2024-01-04","share"),(5,5,"2024-01-05","click"),(6,6,"2024-01-06","view"),(7,7,"2024-01-07","like"),(8,8,"2024-01-08","share")]
user_actions_df = spark.createDataFrame(user_actions_data, ["user_id","action_id","action_date","action_type"])

display(user_actions_df)

window = Window.orderBy("action_date").rowsBetween(-6, 0)

output_df = (
user_actions_df
    .withColumn("rolling_7day_active_users", count(col("user_id")).over(window))
)

display(output_df)




## Que16: Monthly Cohort Retention Analysis

**Difficulty:** Hard

### Problem

A product team wants to measure how well it holds on to customers over time. Group every user into a cohort based on the calendar month of their first-ever transaction. Then, for each later month, count how many users from that cohort transacted again.

Report one row per cohort and month offset with these columns:
- `cohort_month`: the cohort's first-purchase month formatted as `YYYY-MM`.
- `month_number`: the whole number of months between the activity month and the cohort month (0 for the cohort's own first month).
- `active_users`: the count of distinct users from that cohort who transacted in that month.
- `retention_rate`: `active_users` as a percentage of the cohort's original size, rounded to 2 decimals.

A user is counted once per month no matter how many times they transact.

**Schema columns:** `transactions.user_id`, `transactions.transaction_date`, `transactions.amount`

**Output columns:** `cohort_month`, `month_number`, `active_users`, `retention_rate`

Sort by the first column in ascending order.

### Examples

#### Example 1

**Input:**

**transactions:**

| user_id | transaction_date | amount |
|--------:|-----------------|-------:|
| 20240100 | 2024-01-08 | 88.28 |
| 20240100 | 2024-02-08 | 21.66 |
| 20240100 | 2024-03-12 | 95.95 |
| 20240101 | 2024-01-21 | 56.86 |
| 20240101 | 2024-01-31 | 70.68 |
| 20240101 | 2024-03-25 | 73.34 |
| 20240102 | 2024-01-07 | 62.99 |
| 20240200 | 2024-02-25 | 49.50 |
| 20240200 | 2024-03-09 | 27.74 |
| 20240201 | 2024-02-27 | 31.53 |

**Output:**

| cohort_month | month_number | active_users | retention_rate |
|-------------|-------------:|-------------:|---------------:|
| 2024-01 | 0 | 3 | 100.00 |
| 2024-01 | 1 | 1 | 33.33 |
| 2024-01 | 2 | 2 | 66.67 |
| 2024-02 | 0 | 2 | 100.00 |
| 2024-02 | 1 | 1 | 50.00 |

**Explanation:** Users 20240100, 20240101, and 20240102 all first transacted in January, so the 2024-01 cohort has 3 users and shows 100.00 at month 0. In March (month 2), only 20240100 and 20240101 transact again, giving 2 active users and a retention rate of 66.67%.

### Constraints

- A user's cohort is the month of their earliest transaction; month 0 always shows 100% retention.
- `month_number` is the whole-month gap between the activity month and the cohort month.
- Each user counts at most once per month, regardless of transaction count.
- `retention_rate` is a percentage rounded to 2 decimal places.
- Return results matching the expected output schema and order.

In [0]:
transactions_data = [(20240100,"2024-01-08",88.28),(20240100,"2024-02-08",21.66),(20240100,"2024-03-12",95.95),(20240101,"2024-01-21",56.86),(20240101,"2024-01-31",70.68),(20240101,"2024-03-25",73.34),(20240102,"2024-01-07",62.99),(20240200,"2024-02-25",49.50),(20240200,"2024-03-09",27.74),(20240201,"2024-02-27",31.53)]
transactions_df = spark.createDataFrame(transactions_data, ["user_id","transaction_date","amount"])

display(transactions_df)

# 1. Convert transaction_date to date
df = transactions_df.withColumn("transaction_date",to_date("transaction_date"))

# 2. Find each user's first transaction month = cohort_month
user_cohorts = (
df.groupBy("user_id").agg(
    date_trunc("month", min("transaction_date")).alias("cohort_month")
))

# 3. Get the activity month for each transaction
activity = (
df
    .withColumn("activity_month",date_trunc("month", col("transaction_date")))
    .join(user_cohorts, "user_id")
    .select("user_id", "cohort_month", "activity_month")
    .distinct()   # user counted only once per month
)

# 4. Calculate how many months after the cohort month
activity = activity.withColumn(
    "month_number",
    floor(months_between(col("activity_month"), col("cohort_month"))).cast("int")
)

# 5. Calculate original cohort size
cohort_sizes = (
    user_cohorts
    .groupBy("cohort_month")
    .agg(
        countDistinct("user_id").alias("cohort_size")
    )
)

# 6. Count active users for each cohort + month_number
result = (
    activity
    .groupBy("cohort_month", "month_number")
    .agg(
        countDistinct("user_id").alias("active_users")
    )
    .join(cohort_sizes, "cohort_month")
    .withColumn(
        "retention_rate",
        round(
            col("active_users") / col("cohort_size") * 100,
            2
        )
    )
    .select(
        date_format("cohort_month", "yyyy-MM").alias("cohort_month"),
        "month_number",
        "active_users",
        "retention_rate"
    )
    .orderBy("cohort_month", "month_number")
)

display(result)

## Que17: A/B Test Result Analysis

**Difficulty:** Hard

### Problem

A product team ran an A/B test that assigns each user to either the control or the treatment variant, records whether that user converted, and logs a numeric outcome metric for each user. The team wants a one-row summary per variant to compare performance.

For each variant, return the number of users in that variant, the percentage of those users who converted, the average of their metric values, and the spread of their metric values measured as the sample standard deviation.

**Schema columns:** `experiment_data.user_id`, `experiment_data.variant`, `experiment_data.converted`, `experiment_data.metric_value`

**Output columns:** `variant`, `sample_size`, `conversion_rate`, `metric_mean`, `metric_std`

- `sample_size`: the number of users in the variant.
- `conversion_rate`: 100 times the number of converted users divided by `sample_size`.
- `metric_mean`: the average `metric_value` for the variant.
- `metric_std`: the sample (n-1) standard deviation of `metric_value` for the variant.

### Examples

#### Example 1

**Input:**

**experiment_data:**

| user_id | variant | converted | metric_value |
|--------|---------|:---------:|-------------:|
| U001 | control | 1 | 85.5 |
| U002 | control | 0 | 72.3 |
| U003 | control | 1 | 91.2 |
| U004 | control | 1 | 88.7 |
| U005 | control | 0 | 65.4 |
| ... | ... | ... | ... |
| U011 | treatment | 1 | 92.4 |
| U012 | treatment | 1 | 95.1 |
| U013 | treatment | 0 | 78.3 |
| ... | ... | ... | ... |

**Output:**

| variant | sample_size | conversion_rate | metric_mean | metric_std |
|---------|------------:|----------------:|------------:|-----------:|
| control | 12 | 58.33 | 80.21 | 10.01 |
| treatment | 13 | 76.92 | 91.47 | 6.41 |

**Explanation:** The control variant has 12 users with 7 conversions (58.33%), a mean metric of 80.2, and a sample standard deviation of 10.01.

### Constraints

- `conversion_rate` is a percentage between 0 and 100.
- Round `conversion_rate`, `metric_mean`, and `metric_std` to 2 decimal places.
- `metric_std` uses the sample (n-1) standard deviation.
- Order by variant alphabetically ascending (control before treatment).
- Return results matching the expected output schema and order.

In [0]:
experiment_data_data = [("U001","control",1,85.5),("U002","control",0,72.3),("U003","control",1,91.2),("U004","control",1,88.7),("U005","control",0,65.4),("U006","control",1,89.1),("U007","control",0,70.2),("U008","control",1,86.9),("U009","control",1,87.3),("U010","control",0,68.1),("U011","treatment",1,92.4),("U012","treatment",1,95.1),("U013","treatment",0,78.3),("U014","treatment",1,93.7),("U015","treatment",1,96.2),("U016","treatment",1,94.5),("U017","treatment",0,81.2),("U018","treatment",1,94.8),("U019","treatment",1,95.6),("U020","treatment",1,97.3),("U021","treatment",0,82.1),("U022","treatment",1,93.2),("U023","control",0,69.5),("U024","control",1,88.2),("U025","treatment",1,94.7)]
experiment_data_df = spark.createDataFrame(experiment_data_data, ["user_id","variant","converted","metric_value"])

display(experiment_data_df)


output_df = (
experiment_data_df . groupBy("variant").agg(
        count("user_id").alias("sample_size"),
        sum("converted").alias("converted_users"),
        avg("metric_value").alias("metric_mean"),
        stddev_samp("metric_value").alias("metric_std")
)
.withColumn( "conversion_rate", col("converted_users") / col("sample_size") * 100)
.select("variant","sample_size",round("conversion_rate", 2).alias("conversion_rate"),round("metric_mean", 2).alias("metric_mean"),round("metric_std", 2).alias("metric_std"))
.orderBy("variant")
)

display(output_df)

## Que18: Partitioning and Bucketing Analysis

**Difficulty:** Hard

### Problem

You are choosing a column to physically partition an event log by, and you want to compare candidates on how evenly they would spread the data. Evaluate two candidate columns, `event_date` and `region`, and for each report:

- `partition_key`: the name of the candidate column being evaluated (`event_date` or `region`).
- `unique_values`: how many distinct values that column has across the log.
- `total_records`: the total number of rows in the log.
- `avg_records_per_partition`: `total_records` divided by `unique_values`.
- `size_variance`: the sample standard deviation of the per-value row counts.

Result should be ordered by `unique_values` descending, then by `partition_key`.

**Schema columns:** `event_log.event_id`, `event_log.event_date`, `event_log.event_type`, `event_log.user_id`, `event_log.region`, `event_log.payload_size`

**Output columns:** `partition_key`, `unique_values`, `total_records`, `avg_records_per_partition`, `size_variance`

### Examples

#### Example 1

**Input:**

**event_log:**

| event_id | event_date | event_type | user_id | region | payload_size |
|---------:|------------|------------|---------|--------|-----------:|
| 1 | 2024-01-01 | login | U001 | North | 1024 |
| 2 | 2024-01-01 | click | U002 | North | 2048 |
| 3 | 2024-01-01 | purchase | U003 | South | 4096 |
| 4 | 2024-01-02 | view | U004 | South | 512 |
| 5 | 2024-01-02 | login | U005 | East | 1024 |
| 6 | 2024-01-03 | click | U001 | West | 2048 |

**Output:**

| partition_key | unique_values | total_records | avg_records_per_partition | size_variance |
|---------------|-------------:|-------------:|--------------------------:|-------------:|
| region | 4 | 6 | 1.50 | 0.58 |
| event_date | 3 | 6 | 2.00 | 1.00 |

**Explanation:** The 6 rows have 3 distinct `event_date` values, so its `avg_records_per_partition` is 6 / 3 = 2.00. `region` has 4 distinct values giving 6 / 4 = 1.50. `region` sorts first because it has more distinct values.

### Constraints

- Report exactly one row per candidate column: `event_date` and `region`.
- `unique_values` counts distinct values of the candidate column.
- `avg_records_per_partition` and `size_variance` are each rounded to 2 decimals.
- `size_variance` uses the sample standard deviation of the per-value row counts.
- Return results matching the expected output schema and order.

In [0]:
event_log_data = [(1,"2024-01-01","login","U001","North",1024),(2,"2024-01-01","click","U002","North",2048),(3,"2024-01-01","purchase","U003","South",4096),(4,"2024-01-02","view","U004","South",512),(5,"2024-01-02","login","U005","East",1024),(6,"2024-01-03","click","U001","West",2048)]
event_log_df = spark.createDataFrame(event_log_data, ["event_id","event_date","event_type","user_id","region","payload_size"])

display(event_log_df)


from pyspark.sql import functions as F

# Total number of records
total_records = event_log_df.count()

# Evaluate event_date
date_stats = (
    event_log_df
    .groupBy("event_date")
    .count()
)

date_result = (
    date_stats
    .agg(
        F.count("*").alias("unique_values"),
        F.stddev_samp("count").alias("size_variance")
    )
    .withColumn("partition_key", F.lit("event_date"))
    .withColumn("total_records", F.lit(total_records))
    .withColumn(
        "avg_records_per_partition",
        F.col("total_records") / F.col("unique_values")
    )
)

# Evaluate region
region_stats = (
    event_log_df
    .groupBy("region")
    .count()
)

region_result = (
    region_stats
    .agg(
        F.count("*").alias("unique_values"),
        F.stddev_samp("count").alias("size_variance")
    )
    .withColumn("partition_key", F.lit("region"))
    .withColumn("total_records", F.lit(total_records))
    .withColumn(
        "avg_records_per_partition",
        F.col("total_records") / F.col("unique_values")
    )
)

# Combine both candidates
result = (
    date_result
    .unionByName(region_result)
    .select(
        "partition_key",
        "unique_values",
        "total_records",
        F.round("avg_records_per_partition", 2).alias(
            "avg_records_per_partition"
        ),
        F.round("size_variance", 2).alias("size_variance")
    )
    .orderBy(
        F.col("unique_values").desc(),
        F.col("partition_key")
    )
)

display(result)


## Que19: PayPal Disputed Transaction Resolution Time

**Difficulty:** Easy

### Problem

PayPal's customer service team tracks every disputed transaction from the day it is filed to the day it is finally resolved, and leadership wants resolution-time statistics broken down by dispute status. Consider only disputes that have actually been resolved (those with a non-NULL `resolved_date`). For each status, report one row with: `resolution_status`, `avg_resolution_days` (average number of days from `filed_date` to `resolved_date`, rounded to 2 decimal places), `min_days`, and `max_days`. A day count is the whole-day difference `resolved_date - filed_date`. Sort the result by `resolution_status` ascending.

**Schema columns:** `disputes.dispute_id`, `disputes.category`, `disputes.filed_date`, `disputes.resolved_date`, `disputes.amount`, `disputes.status`

**Output columns:** `resolution_status`, `avg_resolution_days`, `min_days`, `max_days`

### Examples

#### Example 1

**Input:**

**disputes:**

| dispute_id | category | filed_date | resolved_date | amount | status |
|-----------:|----------|------------|---------------|-------:|--------|
| 1 | Unauthorized_Access | 2026-01-15 | 2026-01-22 | 150 | resolved |
| 3 | Unauthorized_Access | 2026-01-20 | 2026-01-25 | 200.5 | resolved |
| 7 | Quality_Issues | 2026-02-03 | 2026-02-28 | 95.5 | resolved |
| 16 | Quality_Issues | 2026-03-01 | 2026-03-25 | 130 | pending |
| 17 | Item_Not_Received | 2026-03-05 | 2026-03-20 | 98.5 | pending |
| 18 | Unauthorized_Access | 2026-03-08 | NULL | 250 | pending |

**Output:**

| resolution_status | avg_resolution_days | min_days | max_days |
|-------------------|--------------------:|---------:|---------:|
| pending | 19.50 | 15 | 24 |
| resolved | 12.33 | 5 | 25 |

**Explanation:** Dispute 18 is dropped because it has no `resolved_date`. The three resolved disputes take 7, 5, and 25 days, giving an average of 12.33. The two pending disputes take 24 and 15 days, averaging 19.50.

### Constraints

- Include only disputes where `resolved_date` is not NULL; unresolved disputes are excluded entirely.
- Day counts are whole-day date differences (`resolved_date - filed_date`); same-day resolution counts as 0 days.
- Round `avg_resolution_days` to 2 decimal places; `min_days` and `max_days` are whole integers.
- Sort by `resolution_status` in ascending order.
- Return results matching the expected output schema and order.

In [0]:
disputes_data = [(1,"Unauthorized_Access","2026-01-15","2026-01-22",150.0,"resolved"),(3,"Unauthorized_Access","2026-01-20","2026-01-25",200.5,"resolved"),(7,"Quality_Issues","2026-02-03","2026-02-28",95.5,"resolved"),(16,"Quality_Issues","2026-03-01","2026-03-25",130.0,"pending"),(17,"Item_Not_Received","2026-03-05","2026-03-20",98.5,"pending"),(18,"Unauthorized_Access","2026-03-08",None,250.0,"pending")]
disputes_df = spark.createDataFrame(disputes_data, ["dispute_id","category","filed_date","resolved_date","amount","status"])

disputes_df = disputes_df.dropna(subset=["resolved_date"])

disputes_df = (
disputes_df
    .withColumn("filed_date", to_timestamp(col("filed_date"), "yyyy-MM-dd"))
    .withColumn("resolved_date", to_timestamp(col("resolved_date"), "yyyy-MM-dd"))
    .withColumn("duration", timestamp_diff("day", col("filed_date"), col("resolved_date")))

)

grouped_df = (
disputes_df.groupBy("status").agg(
    round(avg("duration"), 2).alias("avg_resolution_days"),
    min(col("duration")).alias("min_days"),
    max(col("duration")).alias("max_days")
))


display(grouped_df)

## Que20: Temporal Table Query — As Of Timestamp

**Difficulty:** Easy

### Problem

Retrieve the current version of every record from a temporal (slowly changing) table. The `employees_history` table tracks salary history: each row is one version of an employee's record, valid from `valid_from` until `valid_to`. When a record is superseded, its `valid_to` is set to the end date; the currently active version of each employee has no end date (`valid_to` is NULL).

Write a query that returns the current record for every employee — the row whose `valid_to` is NULL. Return `employee_id`, `name`, `salary` and `valid_from`, sorted by `employee_id` ascending.

**Schema columns:** `employees_history.employee_id`, `employees_history.name`, `employees_history.salary`, `employees_history.valid_from`, `employees_history.valid_to`

**Output columns:** `employee_id`, `name`, `salary`, `valid_from`

### Examples

#### Example 1

**Input:**

**employees_history:**

| employee_id | name | salary | valid_from | valid_to |
|------------:|------|-------:|------------|----------|
| 1 | Alice | 50000 | 2022-01-01 | 2023-05-31 |
| 1 | Alice | 55000 | 2023-06-01 | 2023-12-31 |
| 1 | Alice | 60000 | 2024-01-01 | NULL |
| 2 | Bob | 65000 | 2023-01-01 | 2024-02-28 |
| 2 | Bob | 70000 | 2024-03-01 | NULL |
| 3 | Charlie | 55000 | 2022-06-01 | 2024-01-31 |
| 3 | Charlie | 58000 | 2024-02-01 | NULL |
| 4 | David | 72000 | 2023-09-01 | NULL |
| 5 | Eve | 48000 | 2023-03-01 | 2024-04-30 |
| 5 | Eve | 52000 | 2024-05-01 | NULL |

**Output:**

| employee_id | name | salary | valid_from |
|------------:|------|-------:|------------|
| 1 | Alice | 60000 | 2024-01-01 |
| 2 | Bob | 70000 | 2024-03-01 |
| 3 | Charlie | 58000 | 2024-02-01 |
| 4 | David | 72000 | 2023-09-01 |
| 5 | Eve | 52000 | 2024-05-01 |

**Explanation:** Employee 1 (Alice) has three versions, but only the row valid from 2024-01-01 has no end date, so it is her current record with salary 60000.

### Constraints

- A record is current when `valid_to` is NULL; superseded versions have a `valid_to` end date.
- Each employee may have multiple historical versions — return exactly one (current) row per employee.
- Output columns must be exactly `employee_id`, `name`, `salary`, `valid_from`.
- Sort results by `employee_id` ascending.

In [0]:
employees_history_data = [(1,"Alice",50000,"2022-01-01","2023-05-31"),(1,"Alice",55000,"2023-06-01","2023-12-31"),(1,"Alice",60000,"2024-01-01",None),(2,"Bob",65000,"2023-01-01","2024-02-28"),(2,"Bob",70000,"2024-03-01",None),(3,"Charlie",55000,"2022-06-01","2024-01-31"),(3,"Charlie",58000,"2024-02-01",None),(4,"David",72000,"2023-09-01",None),(5,"Eve",48000,"2023-03-01","2024-04-30"),(5,"Eve",52000,"2024-05-01",None)]
employees_history_df = spark.createDataFrame(employees_history_data, ["employee_id","name","salary","valid_from","valid_to"])

display(employees_history_df)

output_df = employees_history_df.filter(col("valid_to").isNull()).select("employee_id", "name", "salary", "valid_from")

display(output_df)

## Que21: Best Rated Hotels by Category

**Difficulty:** Easy

### Problem

A hotel booking site wants to feature its highest-rated properties. From the ratings table, return the three hotels with the highest average score. Report each hotel's name and its average score, ordered from the highest score to the lowest.

**Schema columns:** `trh_top_rated.hotel_address`, `trh_top_rated.average_score`, `trh_top_rated.hotel_name`

**Output columns:** `hotel_name`, `average_score`

### Examples

#### Example 1

**Input:**

**trh_top_rated:**

| hotel_address | average_score | hotel_name |
|---------------|:-------------:|------------|
| 123 Ocean Ave, Miami, FL | 4.2 | Ocean View |
| 456 Mountain Rd, Boulder, CO | 3.9 | Mountain Lodge |
| 789 Downtown St, New York, NY | 4.7 | Central Park Hotel |
| 101 Lakeside Blvd, Austin, TX | 4.0 | Lakeside Inn |
| 202 River Ave, Nashville, TN | 4.5 | Riverside |

**Output:**

| hotel_name | average_score |
|------------|:-------------:|
| Central Park Hotel | 4.7 |
| Riverside | 4.5 |
| Ocean View | 4.2 |

**Explanation:** Ranking all five hotels by score, Central Park Hotel (4.7), Riverside (4.5), and Ocean View (4.2) are the top three.

### Constraints

- Return at most three hotels.
- Order rows by average score from highest to lowest.
- Return results matching the expected output schema and order.

In [0]:
trh_top_rated_data = [("123 Ocean Ave, Miami, FL",4.2,"Ocean View"),("456 Mountain Rd, Boulder, CO",3.9,"Mountain Lodge"),("789 Downtown St, New York, NY",4.7,"Central Park Hotel"),("101 Lakeside Blvd, Austin, TX",4.0,"Lakeside Inn"),("202 River Ave, Nashville, TN",4.5,"Riverside")]
trh_top_rated_df = spark.createDataFrame(trh_top_rated_data, ["hotel_address","average_score","hotel_name"])


window = Window.orderBy(col("average_score").desc())

trh_top_rated_df = trh_top_rated_df.withColumn("dense_rank", dense_rank().over(window))

display(trh_top_rated_df)

output_df = trh_top_rated_df.filter(col("dense_rank") <= 3).select("hotel_name", "average_score")

display(output_df)


## Que22: Project Score Averaging

**Difficulty:** Easy

### Problem

A delivery team records member scores for its projects and may ingest duplicate score rows. For each project represented by more than one distinct team member, return `project_id` and the one-decimal `average_score` after identical project-member-score records are counted once, ordered by project.

**Schema columns:** `cas_compute_average.project_id`, `cas_compute_average.team_member_id`, `cas_compute_average.score`, `cas_compute_average.date`

**Output columns:** `project_id`, `average_score`

Sort the results by `project_id`.

### Examples

#### Example 1

**Input:**

**cas_compute_average:**

| project_id | team_member_id | score | date |
|-----------:|---------------:|------:|------|
| 1 | 10 | 80 | 2024-01-01 |
| 1 | 11 | 90 | 2024-01-02 |
| 1 | 11 | 90 | 2024-01-02 |
| 2 | 20 | 70 | 2024-01-03 |

**Output:**

| project_id | average_score |
|-----------:|--------------:|
| 1 | 85.0 |

**Explanation:** Project 1 keeps the distinct scores 80 and 90, so its average is (80 + 90) / 2 = 85.0. Project 2 has only one member and is excluded.

### Constraints

- Exact duplicate project-member-score records count once.
- Projects with only one distinct member are excluded.
- Return results matching the expected output schema and order.

In [0]:
cas_compute_average_data = [(1,10,80,"2024-01-01"),(1,11,90,"2024-01-02"),(1,11,90,"2024-01-02"),(2,20,70,"2024-01-03")]
cas_compute_average_df = spark.createDataFrame(cas_compute_average_data, ["project_id","team_member_id","score","date"])

cas_compute_average_df = cas_compute_average_df.dropDuplicates(subset=["project_id","team_member_id","score"])

display(cas_compute_average_df)

grouped_df = (
cas_compute_average_df.groupBy("project_id").agg(
    round(avg("score"), 1).alias("average_score"),
    count("team_member_id").alias("team_members")
)
.filter("team_members > 1").drop("team_members")
)

display(grouped_df)


## Que23: Self-Viewing Authors Detection

**Difficulty:** Easy

### Problem

A publishing platform records every article view. Identify authors who viewed at least one of their own articles, return each matching author once as `id`, and order the IDs ascending.

**Schema columns:** `avow_sample.article_id`, `avow_sample.author_id`, `avow_sample.viewer_id`, `avow_sample.view_date`

**Output columns:** `id`

### Examples

#### Example 1

**Input:**

**avow_sample:**

| author_id | view_date | viewer_id | article_id |
|----------:|-----------|----------:|-----------:|
| 3 | 2019-08-01 | 5 | 1 |
| 3 | 2019-08-02 | 6 | 1 |
| 7 | 2019-08-01 | 7 | 2 |
| 7 | 2019-08-02 | 6 | 2 |
| 7 | 2019-07-22 | 1 | 4 |
| 4 | 2019-07-21 | 4 | 3 |
| 4 | 2019-07-21 | 4 | 3 |

**Output:**

| id |
|---:|
| 4 |
| 7 |

**Explanation:** Author 4 appears twice with `viewer_id` 4, but the output includes ID 4 only once; author 7 also qualifies from its self-view row.

### Constraints

- Use every supplied input row when calculating the result.
- Preserve the calculation, filtering, tie handling, and ordering described in the problem.
- Return results matching the expected output schema and order.

In [0]:
avow_sample_data = [(3,"2019-08-01",5,1),(3,"2019-08-02",6,1),(7,"2019-08-01",7,2),(7,"2019-08-02",6,2),(7,"2019-07-22",1,4),(4,"2019-07-21",4,3),(4,"2019-07-21",4,3)]
avow_sample_df = spark.createDataFrame(avow_sample_data, ["author_id","view_date","viewer_id","article_id"])

display(avow_sample_df)

joined_df = (
avow_sample_df.alias("a")
    .join(avow_sample_df.alias("v"), on=col("a.author_id") == col("v.viewer_id"))
)

display(joined_df.select(col("a.author_id").alias("id")).distinct().orderBy("id"))


## Que24: A/B Test Conversion Lift

**Difficulty:** Easy

### Problem

A product team wants to compare each experiment variant with its control group. For every experiment and variant, return the number of distinct assigned users, the number of distinct users with a true conversion, the conversion rate, and the relative lift against that experiment's control. `conversion_rate` is converted users divided by total users; `lift` is (variant conversion_rate - control conversion rate) / control conversion rate, with control lift defined as 0.

**Schema columns:** `experiment_users.user_id`, `experiment_users.experiment_id`, `experiment_users.variant`, `experiment_users.signup_date`, `conversions.user_id`, `conversions.experiment_id`, `conversions.converted`, `conversions.conversion_date`

**Output columns:** `experiment_id`, `variant`, `total_users`, `converted_users`, `conversion_rate`, `lift`

### Examples

#### Example 1

**Input:**

**experiment_users:**

| user_id | experiment_id | variant | signup_date |
|--------:|--------------|---------|-------------|
| 1 | EXP001 | control | 2026-01-01 |
| 2 | EXP001 | control | 2026-01-01 |
| 11 | EXP001 | treatment | 2026-01-01 |
| 12 | EXP001 | treatment | 2026-01-01 |

**conversions:**

| user_id | experiment_id | converted | conversion_date |
|--------:|--------------|:---------:|----------------|
| 1 | EXP001 | true | 2026-01-05 |
| 11 | EXP001 | true | 2026-01-05 |
| 12 | EXP001 | true | 2026-01-06 |

**Output:**

| experiment_id | variant | total_users | converted_users | conversion_rate | lift |
|--------------|---------|------------:|----------------:|----------------:|-----:|
| EXP001 | control | 2 | 1 | 0.5000 | 0.0000 |
| EXP001 | treatment | 2 | 2 | 1.0000 | 1.0000 |

**Explanation:** Treatment converts both assigned users, giving a relative lift of 1.0000 over the 0.5000 control rate.

### Constraints

- Count distinct users so duplicate assignment or conversion rows do not inflate either count.
- Count only conversion records where `converted` is `true`.
- Round `conversion_rate` and `lift` to 4 decimal places.
- Sort by `experiment_id` ascending, then `variant` ascending.
- Return results matching the expected output schema and order.

In [0]:
experiment_users_data = [
    (1, "EXP001", "control", "2026-01-01"),
    (2, "EXP001", "control", "2026-01-01"),
    (11, "EXP001", "treatment", "2026-01-01"),
    (12, "EXP001", "treatment", "2026-01-01")
]

experiment_users_df = spark.createDataFrame(
    experiment_users_data,
    ["user_id", "experiment_id", "variant", "signup_date"]
)

conversions_data = [
    (1, "EXP001", True, "2026-01-05"),
    (11, "EXP001", True, "2026-01-05"),
    (12, "EXP001", True, "2026-01-06")
]

conversions_df = spark.createDataFrame(
    conversions_data,
    ["user_id", "experiment_id", "converted", "conversion_date"]
)

total_users_df = (
    experiment_users_df
    .groupBy("experiment_id", "variant")
    .agg(
        countDistinct("user_id").alias("total_users")
    )
)


converted_users_df = (
    experiment_users_df
    .join(
        conversions_df,
        on=["user_id", "experiment_id"],
        how="left"
    )
    .groupBy("experiment_id", "variant")
    .agg(
        countDistinct(
            when(col("converted") == True, col("user_id"))
        ).alias("converted_users")
    )
)


result_df = (
    total_users_df
    .join(
        converted_users_df,
        on=["experiment_id", "variant"],
        how="left"
    )
    .withColumn(
        "conversion_rate",
        col("converted_users") / col("total_users")
    )
)



control_rate_df = (
    result_df
    .filter(col("variant") == "control")
    .select(
        "experiment_id",
        col("conversion_rate").alias("control_conversion_rate")
    )
)


# -----------------------------
# 5. Calculate lift
# -----------------------------

result_df = (
    result_df
    .join(
        control_rate_df,
        on="experiment_id",
        how="left"
    )
    .withColumn(
        "lift",
        when(
            col("variant") == "control",
            lit(0.0)
        ).otherwise(
            (col("conversion_rate") - col("control_conversion_rate"))
            / col("control_conversion_rate")
        )
    )
)


# -----------------------------
# 6. Final formatting
# -----------------------------

output_df = (
    result_df
    .select(
        "experiment_id",
        "variant",
        "total_users",
        "converted_users",
        round("conversion_rate", 4).alias("conversion_rate"),
        round("lift", 4).alias("lift")
    )
    .orderBy("experiment_id", "variant")
)

display(output_df)

## Que25: IoT Sensor Data Analysis

**Difficulty:** Medium

### Problem

The `SDA_sensor_data` table holds IoT sensor readings, each with a `measurement_value` and a `measurement_time` string formatted `MM/DD/YYYY HH:MM:SS`. For each calendar day, order that day's readings from earliest to latest measurement time and number them starting at 1. Sum the values of the readings that land on odd-numbered positions (1st, 3rd, ...) and, separately, the values on even-numbered positions (2nd, 4th, ...). Return one row per day with `measurement_day`, `odd_sum`, and `even_sum`.

**Schema columns:** `SDA_sensor_data.measurement_id`, `SDA_sensor_data.measurement_value`, `SDA_sensor_data.measurement_time`

**Output columns:** `measurement_day`, `odd_sum`, `even_sum`

Sort the results by `measurement_day`.

### Examples

#### Example 1

**Input:**

**SDA_sensor_data:**

| measurement_id | measurement_value | measurement_time |
|---------------:|------------------:|------------------|
| 131233 | 1109.51 | 07/10/2022 09:00:00 |
| 135211 | 1662.74 | 07/10/2022 11:00:00 |
| 523542 | 1246.24 | 07/10/2022 13:15:00 |
| 143562 | 1124.50 | 07/11/2022 15:00:00 |
| 346462 | 1234.14 | 07/11/2022 16:45:00 |

**Output:**

| measurement_day | odd_sum | even_sum |
|-----------------|--------:|---------:|
| 07/10/2022 00:00:00 | 2355.75 | 1662.74 |
| 07/11/2022 00:00:00 | 1124.50 | 1234.14 |

**Explanation:** On 07/10/2022 the readings ordered by time are positions 1, 2, 3; the odd positions 1 and 3 sum to 1109.51 + 1246.24 = 2355.75 and the even position 2 is 1662.74.

### Constraints

- `measurement_time` is a string formatted `MM/DD/YYYY HH:MM:SS`; a reading belongs to the day given by its date part.
- Positions are determined per day by ordering that day's readings by `measurement_time` ascending, numbering from 1.
- `odd_sum` adds the values at odd positions (1st, 3rd, ...); `even_sum` adds the values at even positions (2nd, 4th, ...).
- Output `measurement_day` as the string `MM/DD/YYYY 00:00:00`.
- Return results matching the expected output schema and order.

In [0]:
SDA_sensor_data_data = [(131233,1109.51,"07/10/2022 09:00:00"),(135211,1662.74,"07/10/2022 11:00:00"),(523542,1246.24,"07/10/2022 13:15:00"),(143562,1124.50,"07/11/2022 15:00:00"),(346462,1234.14,"07/11/2022 16:45:00")]
SDA_sensor_data_df = spark.createDataFrame(SDA_sensor_data_data, ["measurement_id","measurement_value","measurement_time"])

window = Window.partitionBy("measurement_date").orderBy(col("measurement_time"))

SDA_sensor_data_df = (
SDA_sensor_data_df
    .withColumn("measurement_time", date_format(to_timestamp(col("measurement_time"),"MM/dd/yyyy HH:mm:ss"),"yyyy-MM-dd HH:mm:ss"))
    .withColumn("measurement_date", date_format(to_date(col("measurement_time")),"MM/dd/yyyy HH:mm:ss"))
    .withColumn("measurement_rank", row_number().over(window))
)


grouped_df = (
SDA_sensor_data_df.groupBy("measurement_date").agg(
    sum(when(col("measurement_rank") % 2 == 0, col("measurement_value"))).alias("even_sum"),
    sum(when(col("measurement_rank") % 2 == 1, col("measurement_value"))).alias("odd_sum")
))



display(grouped_df)


